# Zero-shot real-noise denoising, round 2: the ceiling of pixel-shuffle, and one step on DMID's noise transformation

Round 1 (`real-noise-zero-shot`) put pixel-shuffle downsampling (PD) plus the one-step ADM posterior mean well below
DMID-d on CC, PolyU and FMDD. PSNR was still rising at the largest step multiplier tried, and camera noise is still
correlated at lag 2 (0.39–0.40). This round:

* **PD path, wider:** PD4 with m up to 2.5, PD8, full resolution with the PD4 noise estimate, and PD4 refined by a
  full-resolution step;
* **NT path:** DMID's released noise-transformed images (`pre-trained.zip`: the per-image optimisation that turns
  real noise into Gaussian-like noise) denoised by our one-step posterior mean instead of DMID's 500 two-step
  samples. It runs at DMID's own step, at continuous steps m·σ_D, and with the horizontal flip.

m is chosen on PolyU images 1–50 (dev) and reported on PolyU 51–100, CC and FMDD (test). Same scoring code as round 1.
Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import hashlib, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from skimage.restoration import estimate_sigma

M_PD4 = [1.25, 1.5, 1.75, 2.0, 2.5]       # PD4: step from m · σ̂(each sub-image)
M_PD8 = [1.0, 1.25, 1.5, 2.0]             # PD8 (64×64 sub-images)
M_FULL = [1.5, 2.0, 2.5, 3.0]             # full resolution: step from m · mean σ̂(PD4 sub-images)
REFINE = [0.3, 0.5]                       # PD4 (m = 2) re-denoised at full resolution at c · σ̂(PD4)
M_NT = [0.7, 0.85, 1.0, 1.15, 1.3, 1.5]   # NT images: step from m · σ_D (DMID's level for that image)
DMID_STEP = {"CC": 20, "PolyU": 8}        # main_for_real.py: diffusion_times 21 / 9 -> t = N - 1; FMDD: find_N(noise_level)
DEV_POLYU = 50
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
GH = "https://github.com/Li-Tong-621/DMID/releases/download/v1.0/"
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": (GH + "Datasets.zip", 367_018_356),
    "dmid_real": (GH + "3-Real_world.zip", 147_392_996),
    "dmid_nt": (GH + "pre-trained.zip", 352_155_893),
}
with ThreadPoolExecutor(4) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min")

## 1 · Data: CC (15), PolyU (100), FMDD (48) from DMID's release, and DMID's noise-transformed images

In [ ]:
def read(z, name, mode="RGB"):
    return np.asarray(Image.open(io.BytesIO(z.read(name))).convert(mode))


PAIRS = {  # dataset: (clean folder, noisy folder, clean suffix, noisy suffix, grayscale)
    "CC": ("Datasets/Realworld/CC/GT/", "Datasets/Realworld/CC/Noisy/", "_mean", "_real", False),
    "PolyU": ("Datasets/Realworld/PolyU/PolyU_png/", "Datasets/Realworld/PolyU/Noisy/", "_mean", "_real", False),
    "FMDD": ("Datasets/Realworld/FMDD/gt/", "Datasets/Realworld/FMDD/raw/", "", "", True),
}
DATA = {}
with zipfile.ZipFile(P["datasets"]) as z:
    names = z.namelist()
    for ds, (cdir, ndir, csuf, nsuf, gray) in PAIRS.items():
        cleans = {os.path.splitext(n.split("/")[-1])[0].removesuffix(csuf): n for n in names if n.startswith(cdir) and not n.endswith("/")}
        noisys = {os.path.splitext(n.split("/")[-1])[0].removesuffix(nsuf): n for n in names if n.startswith(ndir) and not n.endswith("/")}
        stems = sorted(set(cleans) & set(noisys), key=natkey)
        mode = "L" if gray else "RGB"
        DATA[ds] = [(s, read(z, cleans[s], mode), read(z, noisys[s], mode)) for s in stems]
SPLIT = {(ds, s): ("dev" if ds == "PolyU" and k < DEV_POLYU else "test") for ds, v in DATA.items() for k, (s, _, _) in enumerate(v)}

# DMID's noise-transformed images: {str(index): {"noise_image": [1, C, H, W] on the 0-255 scale, "noise_level": σ·255}}.
# Their index follows natsorted clean file names; we match each one to our clean image by the highest PSNR instead.
NT = {}
with zipfile.ZipFile(P["dmid_nt"]) as z:
    for ds, member in (("CC", "pre-trained/CC.pt"), ("PolyU", "pre-trained/Polyu.pt"), ("FMDD", "pre-trained/FMDD.pt")):
        raw = torch.load(io.BytesIO(z.read(member)), map_location="cpu", weights_only=False)
        cleans = torch.stack([torch.from_numpy(np.ascontiguousarray(c)).float().reshape(-1) for _, c, _ in DATA[ds]]).to(DEVICES[0])
        for idx, entry in raw.items():
            img = entry["noise_image"].float().reshape(-1, *entry["noise_image"].shape[-2:])  # [C,H,W], 0-255
            flat = (img.mean(0) if ds == "FMDD" else img.permute(1, 2, 0)).reshape(-1).to(DEVICES[0])
            k = int(((cleans - flat) ** 2).mean(1).argmin())
            NT[(ds, DATA[ds][k][0])] = (img / 255, float(entry["noise_level"]))
print({ds: len(v) for ds, v in DATA.items()}, "NT images matched:", {ds: sum(1 for d, _ in NT if d == ds) for ds in DATA},
      "distinct:", len(NT), "example noise levels:", [round(v[1], 2) for k, v in list(NT.items())[:3]])

## 2 · ADM (fp16, SDPA on contiguous q, k, v) and LPIPS on every GPU

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.transpose(1, 2).contiguous() for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


import lpips

CTX = [dict(dev=d, adm=make_adm(d), lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
ADM_STATE.clear()

## 3 · Estimators

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
ABAR = torch.cumprod(1 - BETAS, 0)
NEG_LOG_ABAR = (-torch.log(ABAR)).numpy()


def matched(sigma01):
    """Continuous step t and abar with abar = 1 / (1 + (2σ)^2)."""
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def sigma_of_step(t):
    """Pixel noise level whose matched step is the integer step t."""
    ab = float(ABAR[t])
    return 0.5 * math.sqrt((1 - ab) / ab)


@torch.no_grad()
def tweedie(model, y, sigmas=None, steps=None):
    """One-step posterior means of y [B,3,H,W] ([0,1] units), embedded at the step matched to sigmas[b] or at integer steps[b]."""
    if steps is None:
        pairs = [matched(float(s)) for s in sigmas]
    else:
        pairs = [(float(t), float(ABAR[int(t)])) for t in steps]
    t = torch.tensor([a for a, _ in pairs], dtype=torch.float32, device=y.device)
    ab = torch.tensor([b for _, b in pairs], dtype=torch.float32, device=y.device).view(-1, 1, 1, 1)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = ab.sqrt() * (2 * yp - 1)
    eps = model(x, t)[:, :3].float()
    return (((x - (1 - ab).sqrt() * eps) / ab.sqrt() + 1) / 2)[..., :h, :w]


def pd_split(y, s):
    """[1,3,H,W] -> [s*s,3,H/s,W/s], sub-image k = (k // s, k % s) offset."""
    return torch.cat([y[..., i::s, j::s] for i in range(s) for j in range(s)])


def pd_merge(subs, s):
    k, c, h, w = subs.shape
    out = torch.zeros(1, c, h * s, w * s, device=subs.device, dtype=subs.dtype)
    for n in range(k):
        out[..., n // s::s, n % s::s] = subs[n]
    return out


def sigma_hat(y):
    """Wavelet MAD noise estimate (skimage), averaged over channels; y [3,H,W] on any device."""
    return float(estimate_sigma(y.permute(1, 2, 0).double().cpu().numpy(), channel_axis=-1, average_sigmas=True))


def pd_tweedie(adm, y, s, m):
    """PD-s, each sub-image at the step matched to m · its own noise estimate; batches of at most 16 sub-images."""
    subs = pd_split(y, s)
    levels = [m * sigma_hat(sub) for sub in subs]
    outs = torch.cat([tweedie(adm, subs[i:i + 16], levels[i:i + 16]) for i in range(0, len(subs), 16)])
    return pd_merge(outs, s)


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(a.shape[0], 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=a.shape[0])[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x, gray):
    """[1,3,H,W] float -> uint8 [C,H,W]; grayscale sets average the three channels first, as DMID does for FMDD."""
    x = x[0].clamp(0, 1)
    if gray:
        x = x.mean(0, keepdim=True)
    return (x * 255).round()


def scores(out_u8, ref_u8, lp):
    a, b = out_u8.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    a3, b3 = (a.expand(3, -1, -1), b.expand(3, -1, -1)) if a.shape[0] == 1 else (a, b)
    with torch.no_grad():
        lpv = lp(a3.float()[None] / 127.5 - 1, b3.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf"), ssim=ssim(a, b), lpips=lpv)

## 4 · Every estimator on every image (each family guarded)

In [ ]:
def run_image(ctx, job):
    ds, k = job
    stem, clean, noisy = DATA[ds][k]
    gray = clean.ndim == 2
    dev, adm = ctx["dev"], ctx["adm"]
    to3 = lambda a: torch.from_numpy(np.repeat(a[..., None], 3, -1) if a.ndim == 2 else a.copy()).permute(2, 0, 1).float()
    y = (to3(noisy) / 255)[None].to(dev)
    ref = torch.from_numpy(clean[None] if gray else clean.transpose(2, 0, 1).copy()).to(dev).float()
    s4 = float(np.mean([sigma_hat(sub) for sub in pd_split(y, 4)]))
    outs, errors = {}, []

    def guard(name, fn):
        try:
            outs.update(fn())
        except Exception:
            errors.append((name, traceback.format_exc(limit=3)))

    guard("pd4", lambda: {f"ADM PD4, m={m:g}": pd_tweedie(adm, y, 4, m) for m in M_PD4})
    guard("pd8", lambda: {f"ADM PD8, m={m:g}": pd_tweedie(adm, y, 8, m) for m in M_PD8})
    guard("full", lambda: {f"ADM full (σ̂ PD4), m={m:g}": tweedie(adm, y, [m * s4]) for m in M_FULL})
    guard("refine", lambda: {f"ADM PD4 m=2 + full-res step at {c:g}σ̂": tweedie(adm, outs["ADM PD4, m=2"], [c * s4]) for c in REFINE})
    if (ds, stem) in NT:
        nt, level = NT[(ds, stem)]
        z = (nt.expand(3, -1, -1) if nt.shape[0] == 1 else nt)[None].to(dev)
        if ds in DMID_STEP:
            step = DMID_STEP[ds]
            s_d = sigma_of_step(step)
        else:  # FMDD: find_N(2 · noise_level / 255) matches the step to the stated level
            s_d = level / 255
            step = int(round(matched(s_d)[0]))
        guard("nt_dmid", lambda: {"NT + one step at DMID's step": tweedie(adm, z, steps=[step])})
        guard("nt", lambda: {f"NT + one step, m={m:g}": tweedie(adm, z, [m * s_d]) for m in M_NT})
        guard("nt_flip", lambda: {"NT + one step ×2 (flip), m=1": (outs["NT + one step, m=1"]
                                                                   + tweedie(adm, z.flip(-1), [s_d]).flip(-1)) / 2})
    rows = [dict(dataset=ds, image=stem, split=SPLIT[(ds, stem)], method=m, **scores(to_u8(o, gray), ref, ctx["lpips"])) for m, o in outs.items()]
    return rows, dict(dataset=ds, image=stem, sigma_pd4=s4), errors


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, failures)."""
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = [(ds, k) for ds in ("CC", "FMDD", "PolyU") for k in range(len(DATA[ds]))]
res, fail = worker_pool(jobs[:2], run_image)   # smoke test before the full run
print("smoke test:", len(res), "ok,", len(fail), "failed;", [e for _, _, errs in res for e in errs][:2])
res2, fail2 = worker_pool(jobs[2:], run_image)
res, fail = res + res2, fail + fail2
SCORES = pd.DataFrame([r for rows, _, _ in res for r in rows])
METHOD_ERRORS = [(s["dataset"], s["image"], name, tb) for _, s, errs in res for name, tb in errs]
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
TIMES["denoising"] = time.time() - t0
print(f"{len(res)}/{len(jobs)} images, {len(fail)} failed images, {len(METHOD_ERRORS)} failed methods, {TIMES['denoising'] / 60:.1f} min")
for e in METHOD_ERRORS[:3]:
    print("METHOD ERROR", e)

## 5 · DMID's released outputs, rescored with the same code

In [ ]:
key = lambda a: hashlib.sha1(np.ascontiguousarray(a).tobytes()).hexdigest()


def as_tensor(a):
    """uint8 [H,W] or [H,W,3] -> float [C,H,W] on the first GPU."""
    return torch.from_numpy(a[None] if a.ndim == 2 else a.transpose(2, 0, 1).copy()).float().to(DEVICES[0])


CLEAN_KEYS = {key(c): (ds, s) for ds, v in DATA.items() for s, c, _ in v}
dmid_rows = []
with zipfile.ZipFile(P["dmid_real"]) as z:
    for n in z.namelist():
        m = re.match(r"3-Real_world/(CC|PolyU|FMDD)-([dp])/denoise/(\d+)_denoise\.png$", n)
        if not m:
            continue
        ds, variant, idx = m.groups()
        mode = "L" if ds == "FMDD" else "RGB"
        clean, den = read(z, f"3-Real_world/{ds}-{variant}/clean/{idx}_clean.png", mode), read(z, n, mode)
        match = CLEAN_KEYS.get(key(clean))
        dmid_rows.append(dict(dataset=ds, image=match[1] if match else f"unmatched {idx}", split=SPLIT.get(match, "test"),
                              method=f"DMID-{variant} (released)", **scores(as_tensor(den), as_tensor(clean), CTX[0]["lpips"])))
DMID = pd.DataFrame(dmid_rows)
print("unmatched DMID clean images:", int(DMID.image.str.startswith("unmatched").sum()))

## 6 · Results

In [ ]:
ALL = pd.concat([SCORES, DMID], ignore_index=True)
ALL.to_csv(f"{OUT}/all_scores.csv", index=False)
mean = ALL.groupby(["method", "dataset"])[["psnr", "ssim", "lpips"]].mean()
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()} (all images)")
    print(mean[metric].unstack().round(4 if metric != "psnr" else 3).to_string())
dev = SCORES[SCORES.split == "dev"].groupby("method").psnr.mean()
families = ("ADM PD4, m=", "ADM PD8, m=", "ADM full (σ̂ PD4), m=", "NT + one step, m=")
chosen = {f: dev[[m for m in dev.index if m.startswith(f)]].idxmax() for f in families if any(m.startswith(f) for m in dev.index)}
print("\nchosen on PolyU 1-50 (dev):", chosen)
test = ALL[ALL.split == "test"]
keep = list(chosen.values()) + [m for m in ALL.method.unique() if not any(m.startswith(f) for f in families)]
print("\ntest split (PolyU 51-100, CC, FMDD), mean PSNR / SSIM / LPIPS")
print(test[test.method.isin(keep)].groupby(["method", "dataset"])[["psnr", "ssim", "lpips"]].mean().round(4).unstack().to_string())
wide = ALL.pivot_table(index=["dataset", "image"], columns="method", values="psnr")
for m in [chosen.get("NT + one step, m="), "NT + one step ×2 (flip), m=1", "NT + one step at DMID's step"]:
    if m in wide:
        d = (wide[m] - wide["DMID-d (released)"]).dropna()
        print(f"\n{m} − DMID-d (released), PSNR: " + ", ".join(f"{ds} {g.mean():+.3f} ({int((g > 0).sum())}/{g.size})" for ds, g in d.groupby(level=0)))
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, chosen_on_dev=chosen, grids=dict(pd4=M_PD4, pd8=M_PD8, full=M_FULL, nt=M_NT, refine=REFINE),
                   failed_images=[j for j, _ in fail], failed_methods=[e[:3] for e in METHOD_ERRORS]), f, indent=1)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)), f"total {(time.time() - T_START) / 60:.1f} min")